# LLM Zoomcamp - Module 5: Monitoring Homework

Solving homework questions using OpenTelemetry to instrument a RAG system.

## Cell 1: Setup & Imports

In [1]:
import os
import sys
import sqlite3
from pathlib import Path
from dotenv import load_dotenv

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

from openai import OpenAI
from gitsource import GithubRepositoryDataReader
from minsearch import Index

load_dotenv()

print("✅ All imports successful!")

✅ All imports successful!


## Cell 2: Load RAG System

In [2]:
# Load course documents
COMMIT = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()

# Import RAG helper
monitoring_dir = Path(".")
sys.path.insert(0, str(monitoring_dir))
from rag_helper import RAGBase

rag = RAGBase(index=index, llm_client=client)

print(f"✅ Loaded {len(documents)} documents")
print("✅ RAG system ready!")

✅ Loaded 72 documents
✅ RAG system ready!


## Cell 3: Initialize OpenTelemetry

In [3]:
# Set up OpenTelemetry with ConsoleSpanExporter
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

print("✅ OpenTelemetry initialized with ConsoleSpanExporter")

✅ OpenTelemetry initialized with ConsoleSpanExporter


## Q1: First Trace - Create RAGTraced Class

In [6]:
class RAGTraced(RAGBase):
    """RAG with OpenTelemetry tracing."""
    
    def __init__(self, *args, tracer=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer or trace.get_tracer("llm-zoomcamp")
    
    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            results = super().search(query, num_results)
            span.set_attribute("result_count", len(results))
            return results
    
    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)
            return response
    
    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            # Extract answer from OpenAI response
            answer = response.choices[0].message.content
            span.set_attribute("answer_length", len(answer))
            return answer

# Create RAGTraced instance
rag_traced = RAGTraced(index=index, llm_client=client, tracer=tracer)
print("✅ RAGTraced class created")

✅ RAGTraced class created


## Cell 5: Run Q1 Query

In [7]:
print("\n" + "="*60)
print("=== Q1: First Trace ===")
print("="*60 + "\n")

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

print(f"\n{'='*60}")
print("Answer:")
print(f"{'='*60}\n{answer}")


=== Q1: First Trace ===

{
    "name": "search",
    "context": {
        "trace_id": "0x7bfb5fdf78675b643d671d16d3c5cbc7",
        "span_id": "0x42d62ffe12caca80",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x950d99ad94f542dc",
    "start_time": "2026-07-19T10:17:16.046658Z",
    "end_time": "2026-07-19T10:17:16.049356Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "result_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d06d28db-5010-4385-b7c2-8b4eb8833fe6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    

## Q2: Capturing Metrics as Span Attributes

In [12]:
# Token costs for gpt-4o-mini
TOKEN_COSTS = {
    "gpt-4o-mini": {
        "input": 0.15 / 1_000_000,   # $0.15 per 1M tokens
        "output": 0.60 / 1_000_000,  # $0.60 per 1M tokens
    },
}

class RAGTracedWithMetrics(RAGBase):
    """RAG with OpenTelemetry tracing and metrics."""
    
    def __init__(self, *args, tracer=None, model="gpt-4o-mini", **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer or trace.get_tracer("llm-zoomcamp")
        self.model = model
        self.costs = TOKEN_COSTS.get(model, TOKEN_COSTS["gpt-4o-mini"])
    
    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            results = super().search(query, num_results)
            span.set_attribute("result_count", len(results))
            return results
    
    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)
            
            # Extract and record token usage
            if hasattr(response, 'usage'):
                # OpenAI uses prompt_tokens and completion_tokens
                input_tokens = response.usage.prompt_tokens
                output_tokens = response.usage.completion_tokens
                
                span.set_attribute("input_tokens", input_tokens)
                span.set_attribute("output_tokens", output_tokens)
                
                # Calculate cost
                input_cost = input_tokens * self.costs["input"]
                output_cost = output_tokens * self.costs["output"]
                total_cost = input_cost + output_cost
                
                span.set_attribute("cost", total_cost)
            
            return response
    
    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            answer = response.choices[0].message.content
            span.set_attribute("answer_length", len(answer))
            return answer

# Create instance with metrics
rag_metrics = RAGTracedWithMetrics(index=index, llm_client=client, tracer=tracer, model="gpt-4o-mini")
print("✅ RAGTracedWithMetrics class created")

✅ RAGTracedWithMetrics class created


In [13]:
print("\n" + "="*60)
print("=== Q2: Running traced query with metrics ===")
print("="*60 + "\n")

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_metrics.rag(query)

print(f"\n{'='*60}")
print("Answer:")
print(f"{'='*60}\n{answer}")


=== Q2: Running traced query with metrics ===

{
    "name": "search",
    "context": {
        "trace_id": "0x0b0488fe0be4475ec5aaee7192ae5c44",
        "span_id": "0x9fddbf46b4ae2464",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x7f54dbabc0e07e41",
    "start_time": "2026-07-19T10:27:23.853054Z",
    "end_time": "2026-07-19T10:27:23.854870Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "result_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d06d28db-5010-4385-b7c2-8b4eb8833fe6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
 

## Q3: Span Timing Analysis

In [14]:
print(\"\\n\" + \"=\"*60)\nprint(\"=== Q3: Span Timing Analysis ===\")\nprint(\"=\"*60)\n\nprint(\"\"\"\nFrom the console output above, find the spans and calculate duration:\n\n  duration_ms = (end_time - start_time) / 1_000_000\n\nExample from output:\n  search span:\n    - start_time: \"2026-07-19T10:16:11.383774Z\"\n    - end_time: \"2026-07-19T10:16:11.386042Z\"\n    - duration: ~2.3ms (very fast - local operation)\n  \n  llm span:\n    - start_time: \"2026-07-19T10:16:11.387240Z\"\n    - end_time: \"2026-07-19T10:16:15.418339Z\"\n    - duration: ~4031ms (4 seconds - API call)\n\nTypically:\n  - search: 1-5ms (local text search)\n  - llm: 500-2000ms (API call to OpenAI)\n\n✓ Q3 Answer: 500-2000ms for LLM calls\n\"\"\")

SyntaxError: unexpected character after line continuation character (568782525.py, line 1)